In [42]:
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder
from concrete.ml.sklearn import (
    LinearSVC,
    LogisticRegression,
    RandomForestClassifier,
    SGDClassifier
)
# from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [43]:
base_path = "/Users/mariaasoltanei/Desktop/FACULTATE/CERCETARE/HealthApp/Backend/Data"
train_path = os.path.join(base_path, "Preprocessing/CSVs/train.csv")
test_path = os.path.join(base_path, "MockData/processedTestDF.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=['Activity', 'ActivityName'])
y_train = train_df['ActivityName']
X_test = test_df.drop(columns=['Activity', 'ActivityName'])
y_test = test_df['ActivityName']
# Scale your features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

In [44]:
def get_model(name="LinearSVC"):
    if name == "LinearSVC":
        return LinearSVC(C=30, dual=False, penalty ='l2')
    elif name == "LogisticRegression":
        return LogisticRegression(n_bits=8)
    elif name == "RandomForestClassifier":
        return RandomForestClassifier(n_bits=8, n_estimators=10, max_depth=5)
    elif name == "SGDClassifier":
        return SGDClassifier(n_bits=8)
    else:
        raise ValueError(f"Unknown model: {name}")

In [45]:
#SEE LINK FOR MORE MODELS: https://github.com/zama-ai/concrete-ml/blob/release/1.9.x/docs/references/api/concrete.ml.sklearn.svm.md#class-linearsvr
# Train model
model = LinearSVC(n_bits=3, C=0.01, tol=1e-8)
model.fit(X_train_scaled, y_train_encoded)

# Clear prediction
y_pred_clear = model.predict(X_test_scaled)
print("Predictions (Clear):", y_pred_clear)

# Compile and predict using FHE
model.compile(X_test_scaled)
y_pred = model.predict(X_test_scaled, fhe="execute")
print("Predictions (FHE):", y_pred)


accuracy = accuracy_score(y_test_encoded, y_pred)
precision = precision_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test_encoded, y_pred, average='weighted', zero_division=0)
print(f"Accuracy: {accuracy}")
# report = classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_, zero_division=0)


# Decode predictions back to strings
# decoded_clear = label_encoder.inverse_transform(y_pred_clear)
# decoded_fhe = label_encoder.inverse_transform(y_pred_fhe)
# print(decoded_clear)
# print(decoded_fhe)

# Show results
# for i, (clear, fhe) in enumerate(zip(decoded_clear, decoded_fhe)):
#     print(f"Sample {i+1}: Clear = {clear}, Encrypted = {fhe}")

Predictions (Clear): [1 1 1 1 5 5 4 5 5 5]
Predictions (FHE): [1 1 1 1 5 5 4 5 5 5]
Accuracy: 0.4


In [46]:
# models = {
#     "LinearSVC": LinearSVC(max_iter=10000),
#     "KNeighbors": KNeighborsClassifier(n_neighbors=5)
# }

# results = []

# for model_name, model in models.items():
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_train)  # Note: predicting on training set for quick comparison

#     accuracy = accuracy_score(y_train, y_pred)
#     precision = precision_score(y_train, y_pred, average='weighted', zero_division=0)
#     recall = recall_score(y_train, y_pred, average='weighted', zero_division=0)
#     f1 = f1_score(y_train, y_pred, average='weighted', zero_division=0)
#     print(f"Model: {model_name}")
#     print(f"Accuracy: {accuracy}")
#     print(f"Precision: {precision}")
#     print(f"Recall: {recall}")
#     print(f"F1 Score: {f1}")

#     results.append({
#         'Model': model_name,
#         'Accuracy': accuracy,
#         'Precision': precision,
#         'Recall': recall,
#         'F1 Score': f1
#     })